In [ ]:
import numpy as np

def box_muller(N):
    """
    Generate N iid standard normal random variables
    using the Box-Muller transform.

    Parameters
    ----------
    N : int
        Number of samples to generate.

    Returns
    -------
    Z : ndarray of shape (N,)
        Standard normal random variables.
    """
    # Number of pairs needed
    m = (N + 1) // 2   # ceil(N/2)

    # Uniform(0,1) draws
    U1 = np.random.rand(m)
    U2 = np.random.rand(m)

    # Transform
    R = np.sqrt(-2 * np.log(U1))
    Theta = 2 * np.pi * U2

    X = R * np.cos(Theta)  # first normal from each pair
    Y = R * np.sin(Theta)  # second normal from each pair

    # Stack and trim
    Z = np.concatenate([X, Y])[:N]
    return Z


box_muller(20)


In [ ]:
import numpy as np

def acceptance_rejection_gaussian_laplace(N):
    """
    Generate N iid standard normal random variables using
    the acceptance-rejection method with Laplace(0,1) proposal.

    Parameters
    ----------
    N : int
        Number of samples to generate.

    Returns
    -------
    samples : ndarray of shape (N,)
        Standard normal random variables.
    """

    samples = np.zeros(N)
    num_accepted = 0

    # Compute C_min = (2 / sqrt(2π)) * exp(1/2)
    C_min = (2 / np.sqrt(2 * np.pi)) * np.exp(0.5)

    while num_accepted < N:
        # Generate Laplace(0,1) random variable via inversion
        U = np.random.rand()
        if U < 0.5:
            X = np.log(2 * U)   # left tail
        else:
            X = -np.log(2 * (1 - U))  # right tail

        # Target density: standard normal
        f_X = (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * X**2)

        # Proposal density: Laplace(0,1)
        hat_f_X = 0.5 * np.exp(-abs(X))

        # Acceptance probability
        acceptance_prob = f_X / (C_min * hat_f_X)

        # Accept/reject
        if np.random.rand() <= acceptance_prob:
            samples[num_accepted] = X
            num_accepted += 1

    return samples

samples = acceptance_rejection_gaussian_laplace(10000)
print(samples[:10])  # first 10 samples

In [ ]:
import numpy as np

def marsaglia_polar(N):
    """
    Generate N iid standard normal random variables
    using the Marsaglia polar method.

    Parameters
    ----------
    N : int
        Number of samples to generate.

    Returns
    -------
    Z : ndarray of shape (N,)
        Standard normal random variables.
    """
    Z = np.zeros(N)
    k = 0

    while k < N:
        # Step 1: draw U,V ~ Unif(-1,1)
        U = 2 * np.random.rand() - 1
        V = 2 * np.random.rand() - 1
        S = U**2 + V**2

        # Step 2: reject if outside unit disk or S=0
        if S <= 0 or S >= 1:
            continue

        # Step 3: transform
        T = np.sqrt(-2 * np.log(S) / S)
        X = U * T
        Y = V * T

        # Step 4: assign results
        Z[k] = X
        k += 1
        if k < N:
            Z[k] = Y
            k += 1

    return Z

kapa=0
sum=0
for x in range(100000):
  kapa+=1
  sum+=np.std(marsaglia_polar(10))

print(sum/kapa)

In [ ]:
import numpy as np

def cauchy(N, mu=0.0, lam=1.0):
    """
    Generate N iid Cauchy-distributed random variables
    using the inversion method.

    Parameters
    ----------
    N : int
        Number of random variables to generate (positive integer).
    mu : float
        Location parameter (real number).
    lam : float
        Scale parameter (positive real number).

    Returns
    -------
    X : ndarray of shape (N,)
        Cauchy-distributed random variables.
    """
    # Generate N uniform(0,1) random variables
    U = np.random.rand(N)

    # Apply the inverse CDF of the Cauchy distribution
    X = mu + lam * np.tan(np.pi * (U - 0.5))
    return X

samples = cauchy(10000, mu=0, lam=1)
print(samples[:10])  # print first 10 samples

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def equidistant_rec2d(n: int, s: float) -> float:
    """
    Approximate the average of ||x||^s over the n×n grid points {(i/n, j/n): i,j=1..n}.
    This mirrors the MATLAB code that sums norm(x)^s over the mesh and divides by n^2.
    """
    A = np.linspace(1/n, 1, n)               # points: 1/n, 2/n, ..., 1
    X, Y = np.meshgrid(A, A, indexing="xy")
    R = np.sqrt(X**2 + Y**2)
    return (R**s).mean()                      # sum(...) / n^2

def exact_value(s: float) -> float:
    """
    Exact value helper matching your MATLAB constants.
    Supported s values: -1.5 and -0.5.
    """
    if np.isclose(s, -1.5):
        return 2 / (s + 2) * 0.830896
    if np.isclose(s, -0.5):
        return 2 / (s + 2) * 0.93749
    raise ValueError("Exact value not provided for this s. Use s = -1.5 or s = -0.5.")

def main():
    # choose s and its exact value (uncomment one)
    s = -1.5
    # s = -0.5
    exact = exact_value(s)

    n_vals = 2 ** np.arange(7, 11)           # 2^(7:10)
    errors = np.zeros_like(n_vals, dtype=float)

    for i, n in enumerate(n_vals):
        approx = equidistant_rec2d(int(n), s)
        errors[i] = abs(approx - exact)

    # plot error vs n (log-log)
    plt.figure()
    plt.loglog(n_vals, errors, linewidth=2, marker='*')
    plt.xlabel('n')
    plt.ylabel('absolute value of error')
    plt.grid(True, which="both", ls=':')
    plt.tight_layout()

    # convergence rate from linear fit: log(error) ~ alpha * log(1/n) + c
    slope = np.polyfit(np.log(1.0 / n_vals), np.log(errors), 1)[0]
    print("Convergence rate: ", slope)

    plt.show()

if __name__ == "__main__":
    main()


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

def exact_value(s: float) -> float:
    """
    Exact value helper matching your MATLAB constants.
    Supported s values: -1.5 and -1.2.
    """
    if np.isclose(s, -1.5):
        return 2 / (s + 2) * 0.830896
    if np.isclose(s, -1.2):
        return 2 / (s + 2) * 0.860545
    raise ValueError("Exact value not provided for this s. Use s = -1.5 or s = -1.2.")

def grad_rec2d(n: int, s: float, b: int | float) -> float:
    """
    Graded rectangular rule in 2D for integrand f(x,y) = ||(x,y)||^s
    on the grid {(i/n, j/n): i,j=1..n} with grading exponent b.

    Mirrors MATLAB:
        A = (1/n : 1/n : 1)
        Aleft = A - 1/n
        A = A.^b; Aleft = Aleft.^b
        WX = A - Aleft
        [X,Y] = meshgrid(A,A)
        M = WX' * WX
        Z = sum(sum( (sqrt(X.^2 + Y.^2).^s) .* M ))
    """
    A = np.linspace(1.0 / n, 1.0, n)         # 1/n, 2/n, ..., 1
    Aleft = A - 1.0 / n                      # 0, 1/n, ..., 1-1/n
    A = A ** b
    Aleft = Aleft ** b
    WX = A - Aleft                           # shape (n,)

    # weights matrix M = WX' * WX (outer product)
    M = np.outer(WX, WX)                     # shape (n,n)

    # grid of graded nodes for function evaluation
    X, Y = np.meshgrid(A, A, indexing="xy")
    R = np.sqrt(X * X + Y * Y)

    # sum over grid of f * weight
    return np.sum((R ** s) * M)

def main():
    tic = time.perf_counter()

    # choose s and exact value (uncomment as needed)
    s = -1.5
    # s = -1.2
    exact = exact_value(s)

    n_vals = 2 ** np.arange(6, 10)           # 2^(6:9) -> [64, 128, 256, 512]
    conv = np.zeros(6, dtype=float)          # grading factor b = 1..6

    for j, b in enumerate(range(1, 7), start=0):
        errors = np.zeros_like(n_vals, dtype=float)
        for i, n in enumerate(n_vals):
            approx = grad_rec2d(int(n), s, b)
            errors[i] = abs(approx - exact)

        # slope of log(error) vs log(1/n)
        slope = np.polyfit(np.log(1.0 / n_vals.astype(float)),
                           np.log(errors), 1)[0]
        conv[j] = slope
        print(f"b = {b}: convergence rate = {slope:.6f}")

    # plot convergence rate vs grading factor b
    plt.figure()
    plt.plot(range(1, 7), conv, marker='o')
    plt.xlabel('Grading factor b')
    plt.ylabel('convergence rate')
    plt.grid(True, ls=':')
    plt.tight_layout()
    plt.show()

    toc = time.perf_counter()
    print(f"Elapsed time: {toc - tic:.3f} s")

if __name__ == "__main__":
    main()


In [ ]:
import numpy as np

def estimate_E_exp_X1(M: int = 10**5, t: float = 1.0, sigma: float = 3.0, seed: int = 0):
    """
    Approximates E[exp(X_1)] for X(t) = 1 + t + sigma * W(t),
    with a Monte Carlo estimate and a 95% CLT confidence interval.

    Parameters
    ----------
    M : int
        Number of Monte Carlo samples.
    t : float
        Time point (t > 0).
    sigma : float
        Volatility parameter (sigma > 0).
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    dict
        {
            "exact": exact value of E[exp(X_1)],
            "estimate": Monte Carlo estimate,
            "ci_95": (lower, upper) two-sided 95% CLT CI,
            "abs_error": |estimate - exact|
        }
    """
    rng = np.random.default_rng(seed)

    # X ~ N(0, t)
    X = np.sqrt(t) * rng.standard_normal(M)

    # Exact value: E[exp(1 + t + sigma * X)] = exp(1 + t + 0.5 * sigma^2 * t)
    exact = np.exp(1 + t + 0.5 * sigma**2 * t)

    # Monte Carlo estimator
    samples = np.exp(1 + t + sigma * X)
    estimate = samples.mean()

    # Population variance (to match MATLAB's var(...,1))
    est_var = samples.var(ddof=0)

    # 95% two-sided z-quantile (sqrt(2)*erfinv(0.95) ≈ 1.9599639845)
    z_975 = 1.959963984540054
    half_width = z_975 * np.sqrt(est_var / M)
    ci_95 = (estimate - half_width, estimate + half_width)

    return {
        "exact": exact,
        "estimate": estimate,
        "ci_95": ci_95,
        "abs_error": abs(estimate - exact),
    }

if __name__ == "__main__":
    out = estimate_E_exp_X1(M=10**5, t=1.0, sigma=3.0, seed=0)
    print("Exact value:       ", out["exact"])
    print("Estimated value:   ", out["estimate"])
    print("Confidence interval:", out["ci_95"])
    print("Absolute error:    ", out["abs_error"])

np.random.standard_normal()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def brownian_motion(T: float, N: int, rng: np.random.Generator | None = None) -> np.ndarray:
    """
    Generate a realization of Brownian motion at mesh points 0, T/N, 2T/N, ..., T.

    Parameters
    ----------
    T : float
        Final time (T > 0).
    N : int
        Number of time steps (N >= 1).
    rng : np.random.Generator | None
        Optional NumPy random generator for reproducibility.

    Returns
    -------
    np.ndarray
        Array of shape (N+1,) with W(0)=0 through W(T).
    """
    if rng is None:
        rng = np.random.default_rng()
    dt = T / N
    increments = rng.normal(loc=0.0, scale=np.sqrt(dt), size=N)
    W = np.concatenate(([0.0], np.cumsum(increments)))
    return W

def brownian_motion_plot() -> None:
    """
    Generates and plots 5 sample paths of Brownian motion over [0, 1] with N = 1000.
    Each path is linearly interpolated on a fine grid.
    """
    # Set random seed for reproducibility (similar to rng('default') in MATLAB)
    rng = np.random.default_rng(0)

    T = 1.0
    N = 1000
    num_paths = 5

    # Time grids
    t_coarse = np.linspace(0.0, T, N + 1)               # grid where BM is defined
    t_fine = np.linspace(0.0, T, 1000 * N + 1)          # fine grid for plotting

    plt.figure()
    for _ in range(num_paths):
        W = brownian_motion(T, N, rng=rng)
        # Linear interpolation onto the fine grid
        W_interp = np.interp(t_fine, t_coarse, W)
        plt.plot(t_fine, W_interp)

    plt.xlabel("t")
    plt.ylabel("W(t)")
    plt.grid(True)
    plt.show()

if __name__ == "__main__":
    brownian_motion_plot()


In [ ]:
import numpy as np

# Set random seed for reproducibility
seed = 12345
np.random.seed(seed)

# Initialize parameters
a = 0.0        # Start of interval
b = 1.0        # End of interval
M = int(1e6)   # Number of sample paths
N_values = 4 * 2**np.arange(0, 7)   # N = 4, 8, 16, ..., 256

# Preallocate results array
# Columns: [N, Estimated E[I], Estimated E[I^2]]
results = np.zeros((len(N_values), 3))

# Loop over values of N
for idx, N in enumerate(N_values):
    h = (b - a) / N
    hs = np.sqrt(h)
    t = np.arange(N) / N   # t_j = j/N for j = 0,...,N-1

    sum_I = 0.0
    sum_I2 = 0.0

    # Monte Carlo simulation
    for _ in range(M):
        eta = np.random.randn(N)      # eta_j ~ N(0,1)
        Delta_W = hs * eta            # increments ΔW_j

        # Brownian path (prepend W(0)=0)
        W = np.concatenate(([0.0], np.cumsum(Delta_W)))

        # Compute integrand t_j * W(t_j)
        integrand = t * W[:-1]

        # Ito integral approximation
        I_k = np.sum(integrand * Delta_W)

        sum_I += I_k
        sum_I2 += I_k**2

    # Store results
    E_I = sum_I / M
    E_I2 = sum_I2 / M
    results[idx] = [N, E_I, E_I2]

# Exact values
exact_EI = 0.0
exact_EI2 = 0.25

# Output results
print("Comparison with Exact Values:")
print(f"  Exact E[I]   = {exact_EI:.5f}")
print(f"  Exact E[I^2] = {exact_EI2:.5f}")

print("\nSummary of Results:")
print("N\tEstimated E[I]\tEstimated E[I^2]")
for row in results:
    print(f"{int(row[0])}\t{row[1]:.5f}\t\t{row[2]:.5f}")


In [ ]:
import numpy as np

def EulerMaruyama(T, xi, W):
    """
    Euler–Maruyama approximation for the SDE:
        dX_t = mu(X_t) dt + sigma(X_t) dW_t
    with
        mu(x)    = log(1 + x^2)
        sigma(x) = x * 1_{x > 0}
    """
    N = W.shape[0] - 1       # number of time steps
    M = W.shape[1]           # number of samples
    Delta = T / N

    mu = lambda x: np.log(1 + x**2)
    sigma = lambda x: x * (x > 0)

    # Y is a vector of length M (one value per sample path)
    Y = np.full(M, xi, dtype=float)

    for i in range(N):
        dW = W[i + 1, :] - W[i, :]     # Brownian increments
        Y = Y + mu(Y) * Delta + sigma(Y) * dW

    return Y


if __name__ == "__main__":
    # Parameters
    N0 = 10          # number of steps on coarsest level
    L  = 5           # number of refinement steps
    M  = 10**6       # number of samples
    T  = 1.0         # final time
    xi = 1.0         # initial condition

    # Generate M sample paths of Brownian motion at the finest grid points
    Nmax = N0 * 2**L
    rng = np.random.default_rng(1234567)
    dW = np.sqrt(T / Nmax) * rng.standard_normal(size=(Nmax, M))
    W = np.zeros((Nmax + 1, M))
    W[1:, :] = np.cumsum(dW, axis=0)

    # Vector to hold differences between approximations
    YTdiff = np.zeros(L)   # differences for levels l = 0..L-1 (compared to next level)

    YT_previous = None
    for l in range(L + 1):   # l = 0,1,...,L
        N = N0 * 2**l
        # indices of points on this coarser grid inside the finest grid
        part =  2**(L - l) * np.arange(N + 1)
        Wpart = W[part, :]   # Brownian paths on this grid

        YT = EulerMaruyama(T, xi, Wpart)

        if l > 0:
            # store L^2 difference between this level and previous level
            YTdiff[l - 1] = np.sqrt(np.mean(np.abs(YT - YT_previous)**2))

        YT_previous = YT

    # Compute strong convergence rate: Delta_l = T / (N0 * 2^l), for l = 0..L-1
    Delta = T / (N0 * 2**np.arange(0, L))
    # linear fit in log–log scale
    r = np.polyfit(np.log(Delta), np.log(YTdiff), 1)
    print("Strong rate of convergence in L^2:", r[0])
